In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE paysim AS
SELECT *
FROM read_csv_auto('../data/paysim_transactions.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
con.execute("""
SELECT COUNT(*) AS total_rows
FROM paysim;
""").df()

,total_rows
0,6362620


In [ ]:
alert_priority_summary = con.execute("""
WITH features AS (
    SELECT
        isFraud,
        CASE WHEN type IN ('TRANSFER','CASH_OUT') THEN 1 ELSE 0 END AS risky_type_flag,
        CASE WHEN amount >= 500000 THEN 1 ELSE 0 END AS high_amount_flag,
        CASE WHEN oldbalanceOrg > 0 AND amount >= oldbalanceOrg * 0.90 THEN 1 ELSE 0 END AS high_depletion_flag,
        CASE WHEN type IN ('TRANSFER','CASH_OUT') AND amount > 0
            AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
        THEN 1 ELSE 0 END AS destination_mismatch_flag
    FROM paysim
),

scored AS (
    SELECT
        *,
        risky_type_flag * 2
        + high_amount_flag * 2
        + high_depletion_flag * 3
        + destination_mismatch_flag * 2 AS fraud_risk_score
    FROM features
),

tiered AS (
    SELECT
        *,
        CASE
            WHEN fraud_risk_score BETWEEN 0 AND 2 THEN 'Low'
            WHEN fraud_risk_score BETWEEN 3 AND 5 THEN 'Medium'
            WHEN fraud_risk_score BETWEEN 6 AND 7 THEN 'High'
            WHEN fraud_risk_score >= 8 THEN 'Critical'
        END AS risk_tier
    FROM scored
)

SELECT
    'Critical only' AS review_strategy,
    COUNT(*) AS alerts_reviewed,
    SUM(isFraud) AS fraud_caught,
    ROUND(SUM(isFraud) * 1.0 / 8213, 4) AS recall,
    ROUND(SUM(isFraud) * 1.0 / COUNT(*), 4) AS precision
FROM tiered
WHERE risk_tier = 'Critical'

UNION ALL

SELECT
    'High + Critical' AS review_strategy,
    COUNT(*) AS alerts_reviewed,
    SUM(isFraud) AS fraud_caught,
    ROUND(SUM(isFraud) * 1.0 / 8213, 4) AS recall,
    ROUND(SUM(isFraud) * 1.0 / COUNT(*), 4) AS precision
FROM tiered
WHERE risk_tier IN ('High', 'Critical')

UNION ALL

SELECT
    'Medium + High + Critical' AS review_strategy,
    COUNT(*) AS alerts_reviewed,
    SUM(isFraud) AS fraud_caught,
    ROUND(SUM(isFraud) * 1.0 / 8213, 4) AS recall,
    ROUND(SUM(isFraud) * 1.0 / COUNT(*), 4) AS precision
FROM tiered
WHERE risk_tier IN ('Medium', 'High', 'Critical');
""").df()

alert_priority_summary

,review_strategy,alerts_reviewed,fraud_caught,recall,precision
0,Critical only,21926,1860.0,0.2265,0.0848
1,High + Critical,271029,6096.0,0.7422,0.0225
2,Medium + High + Critical,2327004,8180.0,0.9960,0.0035


In [6]:
alert_priority_summary.to_csv("../outputs/alert_prioritization_summary.csv", index=False)